#Needs md correction
## Transformations applied to Drivers Dataset 


The notebook below accomplishes the following:

1 Read constructors Table from Bronze layer

2 Only required columns for analytics were retailned (Excluded URL column)

3 Standardized column names using snake_case (ConstructorId)

4 Renamed columns

5 removed duplicates

6 Transformed columns to  title case (nationality)

7 Wrote transformed data into the silver layer- constructors table

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
# Read & Write tables for Races

bronze_table=f"{catalog_name}.{bronze_schema}.drivers"
silver_table=f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df= spark.read.table(bronze_table)

In [0]:
display(drivers_df)

In [0]:
#Keeping Relevant tables
drivers_df=drivers_df.drop("url")

In [0]:
# Standardizing the columns

drivers_df=( drivers_df.withColumnsRenamed(
                                {'driverId':'driver_Id',
                                'dateofBirth':'date_of_Birth'}
))

In [0]:
# Concatenating given and family names
drivers_df= (drivers_df
                    .withColumn("driver_name",
                        F.initcap( F.concat_ws(" ",F.col("name.givenName"),F.col("name.familyName"))))
                    .drop("name")
             

)
display(drivers_df)

In [0]:
# finding Duplicates

'''
dupes= (constructors_df.groupBy(constructors_df.columns)
.count()
.filter(F.col("count")>1))
dupes.show()
'''


In [0]:
#Removing Duplicates

drivers_df_deduped= drivers_df.dropDuplicates()
display(drivers_df_deduped)

In [0]:
#Standardizing Column naming convention
drivers_final_df=(drivers_df_deduped
                                .withColumn("nationality",F.initcap(F.col("nationality")))    
                )

In [0]:
#Writing Final table into Silver Layer

(

drivers_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(silver_table)
)
